# Blok 9: Python – Praca z danymi (analiza i wizualizacja)
## Ćwiczenia (SOLVED) – rozwiązania krok po kroku

Ten notatnik zawiera te same zadania co `cwiczenia_clean.ipynb`, ale z pełnymi rozwiązaniami.


## Opis datasetu

**Nazwa zbioru:** Video Game Sales (vgsales)

**Plik:** `vgsales_clean.csv`

**Kontekst biznesowy:** analiza rynku gier wideo – które platformy/genre/publisherzy dominują, jak zmienia się sprzedaż w czasie i w regionach.

**Kolumny (nazwa → typ → opis biznesowy):**
- `Rank` → int → ranking sprzedaży globalnej (1 = największa)
- `Name` → str → tytuł gry
- `Platform` → str/category → platforma (np. Wii, NES, PS4)
- `Year` → int/float → rok wydania
- `Genre` → str/category → gatunek (np. Action, Sports)
- `Publisher` → str/category → wydawca
- `NA_Sales` → float → sprzedaż w Ameryce Płn. (mln szt.)
- `EU_Sales` → float → sprzedaż w Europie (mln szt.)
- `JP_Sales` → float → sprzedaż w Japonii (mln szt.)
- `Other_Sales` → float → sprzedaż w pozostałych regionach (mln szt.)
- `Global_Sales` → float → sprzedaż globalna (mln szt.)

**Etykiety/kategorie:** `Platform`, `Genre`, `Publisher`.

**Uwaga jakościowa:** nawet jeśli plik nazywa się „clean”, ćwiczymy nawyki: typy, braki, duplikaty, sanity checks (np. spójność sum regionów z `Global_Sales`).


## Importy i ustawienia

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except ImportError:
    _HAS_SEABORN = False

try:
    import polars as pl
    _HAS_POLARS = True
except ImportError:
    _HAS_POLARS = False

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 20)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# Część A. NumPy (rozgrzewka)


### Zadanie 1: NumPy – tworzenie tablic

In [ ]:
# 1) Tablica 1D 1..10
a = np.arange(1, 11)

# 2) Parzyste 0..18
b = np.arange(0, 20, 2)

# 3) Macierz 3x4 zer
m = np.zeros((3, 4))

# 4) Wektor długości 5 jedynek
ones = np.ones(5)

a, b, m, ones

### Zadanie 2: NumPy – operacje matematyczne i wektoryzacja

In [ ]:
x = np.array([2, 4, 6, 8, 10, 12, 14, 16])
y = np.array([1, 1, 2, 3, 5, 8, 13, 21])

suma = x + y
roznica = x - y
iloczyn = x * y

sqrt_x = np.sqrt(x)
log_x1 = np.log(x + 1)

# Wersja pętlą (tylko dla porównania czytelności)
suma_loop = []
for i in range(len(x)):
    suma_loop.append(x[i] + y[i])
suma_loop = np.array(suma_loop)

np.allclose(suma, suma_loop), suma, roznica, iloczyn, sqrt_x, log_x1

### Zadanie 3: NumPy – indeksowanie, slicing i maski boolowskie

In [ ]:
z = np.random.normal(loc=0, scale=1, size=20)

slice_ = z[5:12]  # 5..11
mask = z > 0
z_pos = z[mask]
pct_pos = mask.mean() * 100

z, slice_, z_pos, pct_pos

### Zadanie 4: NumPy – statystyki i `axis`

In [ ]:
sales = np.array([
    [10, 12, 9],
    [4, 6, 8],
    [15, 11, 13],
    [7, 5, 6]
])

total = sales.sum()
col_sums = sales.sum(axis=0)
row_sums = sales.sum(axis=1)
mean_all = sales.mean()
std_all = sales.std()

sales, total, col_sums, row_sums, mean_all, std_all

# Część B. pandas – import, przegląd i operacje podstawowe


### Zadanie 5: Wczytanie danych i „3 ekrany”

In [ ]:
df = pd.read_csv('vgsales_clean.csv')

display(df.head())
df.info()
display(df.describe())

# Obserwacje (przykład):
# - Kolumny sales są floatami (mln szt.), a kategorie (Platform/Genre/Publisher) zwykle jako object.
# - Jeśli są braki, zobaczysz je w info() jako różnice w 'non-null count'.

### Zadanie 6: Audit braków danych

In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
})

display(missing_summary.head(10))

### Zadanie 7: Duplikaty – kontrola ryzyka powtórek

In [ ]:
dup_all = df.duplicated().sum()
dup_key = df.duplicated(subset=['Name', 'Platform', 'Year']).sum()

df_dedup = df.drop_duplicates(subset=['Name', 'Platform', 'Year']).copy()

dup_all, dup_key, df.shape, df_dedup.shape

### Zadanie 8: Selekcja kolumn i podstawowe porządkowanie

In [ ]:
df_small = df[['Name', 'Platform', 'Year', 'Genre', 'Publisher', 'Global_Sales']].copy()
df_small = df_small.rename(columns={'Global_Sales': 'global_sales_mln'})

df_no_rank = df.drop(columns=['Rank']).copy() if 'Rank' in df.columns else df.copy()

display(df_small.head())
display(df_no_rank.head())

### Zadanie 9: loc vs iloc – szybkie drille

In [ ]:
# 1) Pierwsze 7 wierszy i wybrane kolumny
q1 = df.loc[:6, ['Name', 'Platform', 'Global_Sales']]

# 2) Wiersze 10–20 pozycjami i kolumny 0–4
q2 = df.iloc[10:21, 0:5]

# 3) Wszystkie gry na Wii (5 pierwszych)
q3 = df.loc[df['Platform'] == 'Wii'].head(5)

# 4) Gry po 2010 z Global_Sales > 5
q4 = df.loc[(df['Year'] > 2010) & (df['Global_Sales'] > 5), ['Name', 'Platform', 'Year', 'Global_Sales']]

# 5) Pojedyncza wartość Genre dla wiersza indeks 0
q5 = df.loc[0, 'Genre']

display(q1)
display(q2)
display(q3)
display(q4.head(10))
q5

### Zadanie 10: Filtrowanie i sortowanie – top sprzedawcy

In [ ]:
top10_a = df.nlargest(10, 'Global_Sales')[['Name', 'Platform', 'Year', 'Global_Sales']]
top10_b = df.sort_values('Global_Sales', ascending=False).head(10)[['Name', 'Platform', 'Year', 'Global_Sales']]

display(top10_a)
display(top10_b)

# Czy identyczne? (uwaga: indeksy mogą być inne, więc porównujemy wartości)
same_values = top10_a.reset_index(drop=True).equals(top10_b.reset_index(drop=True))
same_values

# Część C. EDA w pandas


### Zadanie 11: Feature engineering – udziały regionów

In [ ]:
denom = df['Global_Sales'].replace(0, pd.NA)

df['NA_share'] = df['NA_Sales'] / denom
df['EU_share'] = df['EU_Sales'] / denom
df['JP_share'] = df['JP_Sales'] / denom
df['Other_share'] = df['Other_Sales'] / denom

shares = ['NA_share', 'EU_share', 'JP_share', 'Other_share']
display(df[shares].describe())

# Szybka kontrola zakresu (powinno być ~0..1, poza wyjątkami)
df[shares].agg(['min', 'max'])

### Zadanie 12: Sanity check – czy Global ≈ suma regionów?

In [ ]:
regions_sum = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']
diff = df['Global_Sales'] - regions_sum
abs_diff = diff.abs()

max_abs_diff = abs_diff.max()
count_problem = (abs_diff > 1e-6).sum()

max_abs_diff, count_problem

In [ ]:
worst = df.assign(regions_sum=regions_sum, diff=diff, abs_diff=abs_diff)
worst = worst.sort_values('abs_diff', ascending=False)
display(worst[['Name', 'Platform', 'Year', 'Global_Sales', 'regions_sum', 'diff', 'abs_diff']].head(10))

### Zadanie 13: Rozkład kategorii – Platform i Genre

In [ ]:
vc_platform = df['Platform'].value_counts()
vc_genre = df['Genre'].value_counts()

share_top5 = df['Platform'].value_counts(normalize=True).head(5) * 100

display(vc_platform.head(10))
display(vc_genre.head(10))
share_top5

### Zadanie 14: Groupby – sprzedaż po platformach

In [ ]:
by_platform = (
    df.groupby('Platform', as_index=False)
      .agg(
          global_sales_sum=('Global_Sales', 'sum'),
          games=('Name', 'count'),
          global_sales_mean=('Global_Sales', 'mean')
      )
      .sort_values('global_sales_sum', ascending=False)
)

display(by_platform.head(10))

### Zadanie 15: Trend w czasie – sprzedaż roczna

In [ ]:
by_year = (
    df.groupby('Year', as_index=False)
      .agg(global_sales_sum=('Global_Sales', 'sum'), games=('Name', 'count'))
      .sort_values('Year')
)

display(by_year.head())
display(by_year.tail())

Wnioski (przykład):
- Widać okres wzrostu sprzedaży do pewnego momentu, a potem spadek (często związany z cyklami konsol).
- Lata z najwyższą sprzedażą to potencjalny „peak” rynku w danych – warto sprawdzić, co wtedy dominowało (platformy/genre).


### Zadanie 16: Pivot table – Genre x region (preferencje)

In [ ]:
genre_region = pd.pivot_table(
    df,
    values=['NA_Sales', 'EU_Sales', 'JP_Sales'],
    index='Genre',
    aggfunc='sum',
    fill_value=0
)

top_na = genre_region.sort_values('NA_Sales', ascending=False).head(5)
top_jp = genre_region.sort_values('JP_Sales', ascending=False).head(5)

display(top_na)
display(top_jp)

### Zadanie 17: Top wydawcy i „long tail”

In [ ]:
by_publisher = (
    df.groupby('Publisher', as_index=False)
      .agg(global_sales_sum=('Global_Sales', 'sum'), games=('Name', 'count'))
      .sort_values('global_sales_sum', ascending=False)
)

top10_pub = by_publisher.head(10)
total_sales = df['Global_Sales'].sum()
share_top10 = top10_pub['global_sales_sum'].sum() / total_sales * 100

display(top10_pub)
share_top10

### Zadanie 18: Korelacje regionów

In [ ]:
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
corr = df[sales_cols].corr()
display(corr)

if _HAS_SEABORN:
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True)
    plt.title('Korelacje sprzedaży (regiony)')
    plt.tight_layout()
    plt.show()


Interpretacja (przykład):
- Wysoka korelacja `Global_Sales` z regionami jest naturalna, bo global to suma regionów.
- Korelacje między regionami (NA vs EU vs JP) mogą sugerować podobieństwo gustów, ale to nie jest dowód przyczynowości.


# Część D. Czyszczenie danych


### Zadanie 19: Wymuszenie typów (type enforcement)

In [ ]:
# 1) Year jako numeryczne (jeśli już jest, to ta operacja nic nie popsuje)
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# 2) Ile NaN w Year
n_year_nan = df['Year'].isna().sum()

# 3) Kategorie
for c in ['Platform', 'Genre', 'Publisher']:
    if c in df.columns:
        df[c] = df[c].astype('category')

n_year_nan, df.dtypes

### Zadanie 20: Strategia dla braków

In [ ]:
df_drop = df.dropna(subset=['Year']).copy()

year_median = df['Year'].median()
df_impute = df.copy()
df_impute['Year'] = df_impute['Year'].fillna(year_median)

comparison = pd.DataFrame({
    'version': ['drop', 'impute'],
    'rows': [df_drop.shape[0], df_impute.shape[0]],
    'total_global_sales': [df_drop['Global_Sales'].sum(), df_impute['Global_Sales'].sum()]
})

display(comparison)

# Do trendu w czasie zwykle lepsze jest df_drop (nie zmyślasz dat),
# chyba że braki są małe i imputacja jest uzasadniona biznesowo.

### Zadanie 21: Reguły biznesowe i walidacja

In [ ]:
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

# 1) Ujemne wartości sprzedaży
negative_mask = (df[sales_cols] < 0).any(axis=1)
negatives = df.loc[negative_mask, ['Name', 'Platform', 'Year'] + sales_cols]

# 2) Year poza zakresem (przykładowo 1980–2020)
year_outside = (df['Year'] < 1980) | (df['Year'] > 2020)
years_out = df.loc[year_outside, ['Name', 'Platform', 'Year']].head(20)

# 3) sales_ok
df['sales_ok'] = (~negative_mask) & (~year_outside) & (df['Global_Sales'] >= 0)

display(negatives.head(10))
display(years_out)
df['sales_ok'].value_counts()

# Część E. Wizualizacja


### Zadanie 22: Bar chart – top platformy (sprzedaż)

In [ ]:
top_platform = by_platform.head(10).copy()

plt.figure(figsize=(10, 5))
plt.bar(top_platform['Platform'].astype(str), top_platform['global_sales_sum'])
plt.title('Top 10 platform – suma sprzedaży globalnej')
plt.xlabel('Platforma')
plt.ylabel('Global_Sales (mln szt.)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Zadanie 23: Line plot – trend roczny sprzedaży

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(by_year['Year'], by_year['global_sales_sum'])
plt.title('Sprzedaż globalna per rok')
plt.xlabel('Rok')
plt.ylabel('Global_Sales (mln szt.)')
plt.tight_layout()

# Opcjonalnie: oznaczenie maksimum
idx_max = by_year['global_sales_sum'].idxmax()
max_year = by_year.loc[idx_max, 'Year']
max_val = by_year.loc[idx_max, 'global_sales_sum']
plt.scatter([max_year], [max_val])
plt.annotate(f"max: {int(max_year)}", (max_year, max_val))

plt.show()

### Zadanie 24: Rozkład sprzedaży – histogram i/lub boxplot

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['Global_Sales'], bins=50)
plt.title('Rozkład Global_Sales')
plt.xlabel('Global_Sales (mln)')
plt.ylabel('Liczba gier')
plt.tight_layout()
plt.show()

# Wersja logarytmiczna przez log1p
plt.figure(figsize=(8, 4))
plt.hist(np.log1p(df['Global_Sales']), bins=50)
plt.title('Rozkład log1p(Global_Sales)')
plt.xlabel('log1p(Global_Sales)')
plt.ylabel('Liczba gier')
plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(6, 3))
plt.boxplot(df['Global_Sales'], vert=False)
plt.title('Boxplot Global_Sales')
plt.xlabel('Global_Sales (mln)')
plt.tight_layout()
plt.show()

### Zadanie 25: Storytelling – porównanie preferencji regionów
Wariant prosty i czytelny: 3 osobne wykresy bar dla NA/EU/JP na top gatunkach globalnie.


In [ ]:
# Przygotowanie: sprzedaż globalna per Genre (z regionów)
genre_region_full = genre_region.copy()
genre_region_full['sum_3'] = genre_region_full['NA_Sales'] + genre_region_full['EU_Sales'] + genre_region_full['JP_Sales']

top_genres = genre_region_full.sort_values('sum_3', ascending=False).head(8).index.tolist()
plot_df = genre_region_full.loc[top_genres, ['NA_Sales', 'EU_Sales', 'JP_Sales']].sort_values('NA_Sales', ascending=False)

for col in ['NA_Sales', 'EU_Sales', 'JP_Sales']:
    plt.figure(figsize=(10, 4))
    plt.bar(plot_df.index.astype(str), plot_df[col])
    plt.title(f'Top gatunki – {col.replace("_Sales", "")}')
    plt.xlabel('Genre')
    plt.ylabel('Sales (mln)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_df

Wnioski (przykład):
- NA i EU często mają podobne top gatunki, ale ich kolejność może się różnić.
- JP potrafi wyraźnie preferować inne gatunki (warto sprawdzić, które i o ile).
- To jest punkt startowy do decyzji marketingowych: targetowanie regionów inną ofertą.


# Część F. Łączenie danych (merge) + mini-case


### Zadanie 26: Tabela mapująca (platform → brand_family)

In [ ]:
platform_map = pd.DataFrame({
    'Platform': ['Wii', 'NES', 'SNES', 'N64', 'GC', 'DS', '3DS', 'GB', 'GBA', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'X360', 'XB', 'XOne', 'PC'],
    'brand_family': ['Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Nintendo',
                     'Sony', 'Sony', 'Sony', 'Sony', 'Sony', 'Sony',
                     'Microsoft', 'Microsoft', 'Microsoft',
                     'PC']
})

df_with_brand = df.merge(platform_map, on='Platform', how='left')

n_missing_brand = df_with_brand['brand_family'].isna().sum()
missing_platforms = (
    df_with_brand.loc[df_with_brand['brand_family'].isna(), 'Platform']
    .astype(str)
    .value_counts()
    .head(15)
)

n_missing_brand, missing_platforms

### Zadanie 27: Mini-case – sprzedaż per brand_family

In [ ]:
dfb = df_with_brand.copy()
dfb['brand_family'] = dfb['brand_family'].fillna('Other')

by_brand = (
    dfb.groupby('brand_family', as_index=False)
       .agg(global_sales_sum=('Global_Sales', 'sum'), games=('Name', 'count'))
       .sort_values('global_sales_sum', ascending=False)
)
by_brand['market_share_pct'] = by_brand['global_sales_sum'] / by_brand['global_sales_sum'].sum() * 100

display(by_brand)

plt.figure(figsize=(8, 4))
plt.bar(by_brand['brand_family'].astype(str), by_brand['global_sales_sum'])
plt.title('Sprzedaż globalna per brand_family')
plt.xlabel('brand_family')
plt.ylabel('Global_Sales (mln)')
plt.tight_layout()
plt.show()

Wnioski (przykład):
- Agregacja po rodzinach marek upraszcza obraz rynku z kilkudziesięciu platform do kilku segmentów.
- Udziały procentowe pomagają w szybkiej ocenie dominacji (kto ma „większość tortu”).


# Część G. Mini-projekt: Market snapshot


### Zadanie 28: Market snapshot (1 slajd)
Poniżej przykładowa realizacja: KPI + 2 wykresy + krótkie wnioski.


In [ ]:
total_global_sales = df['Global_Sales'].sum()
n_games = df.shape[0]

top_platform_row = by_platform.iloc[0]
top_platform_name = str(top_platform_row['Platform'])
top_platform_sales = float(top_platform_row['global_sales_sum'])

print('KPI:')
print('1) Total Global Sales (mln):', round(total_global_sales, 2))
print('2) Number of games:', n_games)
print('3) Top platform:', top_platform_name, '| sales (mln):', round(top_platform_sales, 2))


In [ ]:
# Wykres 1: top 10 platform
top_platform = by_platform.head(10).copy()
plt.figure(figsize=(10, 5))
plt.bar(top_platform['Platform'].astype(str), top_platform['global_sales_sum'])
plt.title('Top 10 platform – sprzedaż globalna')
plt.xlabel('Platforma')
plt.ylabel('Global_Sales (mln)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Wykres 2: trend roczny
plt.figure(figsize=(10, 5))
plt.plot(by_year['Year'], by_year['global_sales_sum'])
plt.title('Trend sprzedaży globalnej w czasie')
plt.xlabel('Rok')
plt.ylabel('Global_Sales (mln)')
plt.tight_layout()
plt.show()

Wnioski (przykład, maks 5 zdań):
1) Sprzedaż globalna jest silnie skoncentrowana – top platformy generują dużą część rynku.
2) W trendzie rocznym widać cykliczność (związana m.in. z generacjami konsol) i okresy wzrostu/spadku.
3) Preferencje regionów różnią się – JP częściej ma inny profil gatunków niż NA/EU.
4) Top wydawcy tworzą „long tail”: kilka firm robi znaczną część sprzedaży.
5) Sanity check sum regionów vs global sugeruje, że dane są spójne (różnice głównie z zaokrągleń).


# Część H. Polars (intro)


### Zadanie 29: Polars – szybkie EDA

In [ ]:
if not _HAS_POLARS:
    print('Polars nie jest zainstalowany – pomiń tę część lub zainstaluj: pip install polars')
else:
    pl_df = pl.read_csv('vgsales_clean.csv')

    top_pl = (
        pl_df.group_by('Platform')
             .agg(pl.col('Global_Sales').sum().alias('global_sales_sum'))
             .sort('global_sales_sum', descending=True)
             .head(10)
    )

    by_year_pl = (
        pl_df.group_by('Year')
             .agg(pl.col('Global_Sales').sum().alias('global_sales_sum'))
             .sort('Year')
             .head(10)
    )

    print(top_pl)
    print(by_year_pl)


## Bonus – przykładowe rozwiązania 2 mikro-zadań
(Resztę zostawiam jako dodatkową praktykę.)


In [ ]:
# Bonus 1) Top 10 gier osobno dla NA i JP i overlap
top_na = df.nlargest(10, 'NA_Sales')[['Name', 'Platform', 'NA_Sales']]
top_jp = df.nlargest(10, 'JP_Sales')[['Name', 'Platform', 'JP_Sales']]

set_na = set(top_na['Name'].astype(str))
set_jp = set(top_jp['Name'].astype(str))
overlap = set_na.intersection(set_jp)

display(top_na)
display(top_jp)
print('Overlap (wspólne tytuły):', len(overlap))
print(sorted(list(overlap))[:20])

In [ ]:
# Bonus 5) Scatter NA vs EU + komentarz
plt.figure(figsize=(6, 5))
plt.scatter(df['NA_Sales'], df['EU_Sales'], alpha=0.3)
plt.title('NA_Sales vs EU_Sales')
plt.xlabel('NA_Sales (mln)')
plt.ylabel('EU_Sales (mln)')
plt.tight_layout()
plt.show()

# Komentarz (przykład):
# Jeśli widać dodatnią zależność, to gry popularne w NA często są też popularne w EU,
# ale nadal są wyjątki (hity regionalne).